# Error Analysis - Deep Dive into Misclassifications

Detailed analysis of KNN misclassifications to understand model limitations and potential improvements.

## Contents
1. Load Data and Predictions
2. Error Pattern Analysis
3. Temporal Characteristics of Errors
4. Feature Space Analysis
5. Nearest Neighbor Investigation
6. Recommendations for Improvement

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sys.path.insert(0, str(Path.cwd() / 'src'))
from run_csth import load_csth_dataset
from knn_pipeline import TimeSeriesPreprocessor, PipelineConfig, pairwise_euclidean_matrix

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Load Data and Predictions

In [ ]:
# Load dataset
data_dir = Path('data/raw')
splits = load_csth_dataset(data_dir)

X_train, y_train = splits['train']
X_test, y_test = splits['test']

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Load predictions
results_dir = Path('results')
pred_files = list(results_dir.glob('test_predictions_k*.csv'))

if pred_files:
    pred_file = sorted(pred_files)[-1]
    df_pred = pd.read_csv(pred_file)
    print(f"\nLoaded predictions from: {pred_file.name}")
    print(f"Total predictions: {len(df_pred)}")
    
    # Add error indicators
    df_pred['correct'] = df_pred['true_label'] == df_pred['predicted_label']
    df_pred['error_type'] = 'Correct'
    df_pred.loc[(df_pred['true_label'] == 0) & (df_pred['predicted_label'] == 1), 'error_type'] = 'False Positive'
    df_pred.loc[(df_pred['true_label'] == 1) & (df_pred['predicted_label'] == 0), 'error_type'] = 'False Negative'
    
    n_errors = (~df_pred['correct']).sum()
    print(f"Errors: {n_errors} ({n_errors/len(df_pred)*100:.2f}%)")
else:
    print("\nNo prediction files found. Please run final evaluation first.")
    df_pred = None

## 2. Error Pattern Analysis

In [ ]:
if df_pred is not None:
    # Error type breakdown
    error_summary = df_pred['error_type'].value_counts()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Pie chart of errors
    axes[0].pie(error_summary.values, labels=error_summary.index, 
                autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Error Type Distribution', fontweight='bold', fontsize=12)
    
    # Confidence by error type
    for error_type in ['Correct', 'False Positive', 'False Negative']:
        mask = df_pred['error_type'] == error_type
        if mask.sum() > 0:
            axes[1].hist(df_pred[mask]['confidence'], bins=20, 
                        alpha=0.6, label=error_type, edgecolor='black')
    axes[1].set_xlabel('Confidence')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Confidence Distribution by Error Type', fontweight='bold', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Box plot of confidence
    error_types = df_pred['error_type'].unique()
    data_to_plot = [df_pred[df_pred['error_type'] == et]['confidence'].values 
                    for et in error_types]
    bp = axes[2].boxplot(data_to_plot, labels=error_types, patch_artist=True)
    colors = ['lightgreen', 'lightcoral', 'lightyellow']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    axes[2].set_ylabel('Confidence')
    axes[2].set_title('Confidence by Error Type', fontweight='bold', fontsize=12)
    axes[2].tick_params(axis='x', rotation=15)
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistical analysis
    print("\nConfidence Statistics by Error Type:")
    print("="*80)
    for error_type in error_types:
        mask = df_pred['error_type'] == error_type
        conf = df_pred[mask]['confidence']
        print(f"{error_type:20s}: Mean={conf.mean():.4f}, Std={conf.std():.4f}, "
              f"Median={conf.median():.4f}, Min={conf.min():.4f}, Max={conf.max():.4f}")

In [ ]:
if df_pred is not None:
    # Identify low-confidence errors and high-confidence errors
    errors = df_pred[~df_pred['correct']].copy()
    
    print("\nLow Confidence Errors (Confidence < 0.6):")
    print("="*80)
    low_conf_errors = errors[errors['confidence'] < 0.6].sort_values('confidence')
    if len(low_conf_errors) > 0:
        print(f"Count: {len(low_conf_errors)} ({len(low_conf_errors)/len(errors)*100:.1f}% of errors)")
        print("\nTop 10:")
        for idx, row in low_conf_errors.head(10).iterrows():
            print(f"Sample {row['sample_id']:4d}: {row['error_type']:15s} | "
                  f"Confidence: {row['confidence']:.3f} | True: {row['true_label']}, Pred: {row['predicted_label']}")
    else:
        print("None found.")
    
    print("\n\nHigh Confidence Errors (Confidence > 0.8):")
    print("="*80)
    high_conf_errors = errors[errors['confidence'] > 0.8].sort_values('confidence', ascending=False)
    if len(high_conf_errors) > 0:
        print(f"Count: {len(high_conf_errors)} ({len(high_conf_errors)/len(errors)*100:.1f}% of errors)")
        print("\nTop 10:")
        for idx, row in high_conf_errors.head(10).iterrows():
            print(f"Sample {row['sample_id']:4d}: {row['error_type']:15s} | "
                  f"Confidence: {row['confidence']:.3f} | True: {row['true_label']}, Pred: {row['predicted_label']}")
    else:
        print("None found.")

## 3. Temporal Characteristics of Errors

In [ ]:
if df_pred is not None:
    # Get samples that were misclassified
    error_indices = df_pred[~df_pred['correct']]['sample_id'].values
    correct_indices = df_pred[df_pred['correct']]['sample_id'].values
    
    X_errors = X_test[error_indices]
    y_errors = y_test[error_indices]
    
    X_correct = X_test[correct_indices[:len(error_indices)]]  # Same number for comparison
    y_correct = y_test[correct_indices[:len(error_indices)]]
    
    feature_names = ['Cold Water Flow', 'Tank Level', 'Temperature']
    
    # Plot average patterns
    fig, axes = plt.subplots(3, 3, figsize=(18, 12))
    
    for feat_idx in range(3):
        # Errors
        ax = axes[feat_idx, 0]
        for sample in X_errors[:5]:  # Plot first 5 errors
            ax.plot(sample[:, feat_idx], alpha=0.6, linewidth=1.5)
        ax.set_title(f'Errors - {feature_names[feat_idx]}', fontweight='bold')
        ax.set_xlabel('Time Step')
        ax.grid(True, alpha=0.3)
        
        # Correct predictions
        ax = axes[feat_idx, 1]
        for sample in X_correct[:5]:
            ax.plot(sample[:, feat_idx], alpha=0.6, linewidth=1.5, color='green')
        ax.set_title(f'Correct - {feature_names[feat_idx]}', fontweight='bold')
        ax.set_xlabel('Time Step')
        ax.grid(True, alpha=0.3)
        
        # Average comparison
        ax = axes[feat_idx, 2]
        mean_error = X_errors[:, :, feat_idx].mean(axis=0)
        std_error = X_errors[:, :, feat_idx].std(axis=0)
        mean_correct = X_correct[:, :, feat_idx].mean(axis=0)
        std_correct = X_correct[:, :, feat_idx].std(axis=0)
        
        timesteps = np.arange(len(mean_error))
        ax.plot(timesteps, mean_error, label='Errors', linewidth=2)
        ax.fill_between(timesteps, mean_error - std_error, mean_error + std_error, alpha=0.3)
        ax.plot(timesteps, mean_correct, label='Correct', linewidth=2, color='green')
        ax.fill_between(timesteps, mean_correct - std_correct, mean_correct + std_correct, 
                       alpha=0.3, color='green')
        ax.set_title(f'Average Pattern - {feature_names[feat_idx]}', fontweight='bold')
        ax.set_xlabel('Time Step')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if df_pred is not None:
    # Compute temporal statistics
    print("\nTemporal Variance Analysis (Errors vs Correct):")
    print("="*80)
    
    for feat_idx, feat_name in enumerate(feature_names):
        var_errors = X_errors[:, :, feat_idx].var(axis=1).mean()
        var_correct = X_correct[:, :, feat_idx].var(axis=1).mean()
        
        mean_errors = X_errors[:, :, feat_idx].mean()
        mean_correct = X_correct[:, :, feat_idx].mean()
        
        print(f"\n{feat_name}:")
        print(f"  Temporal Variance (Errors):  {var_errors:.6f}")
        print(f"  Temporal Variance (Correct): {var_correct:.6f}")
        print(f"  Ratio: {var_errors/var_correct:.2f}x")
        print(f"  Mean Value (Errors):  {mean_errors:.4f}")
        print(f"  Mean Value (Correct): {mean_correct:.4f}")

## 4. Feature Space Analysis

In [ ]:
if df_pred is not None:
    # Flatten sequences for analysis
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    
    # Apply PCA for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_test_flat)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # PCA plot colored by prediction correctness
    correct_mask = df_pred['correct'].values
    axes[0].scatter(X_pca[correct_mask, 0], X_pca[correct_mask, 1], 
                   alpha=0.5, s=30, label='Correct', c='green')
    axes[0].scatter(X_pca[~correct_mask, 0], X_pca[~correct_mask, 1],
                   alpha=0.7, s=50, label='Error', c='red', marker='x')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
    axes[0].set_title('PCA: Correct vs Errors', fontweight='bold', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # PCA plot colored by true label with errors highlighted
    fp_mask = (df_pred['error_type'] == 'False Positive').values
    fn_mask = (df_pred['error_type'] == 'False Negative').values
    
    # Plot all points first
    axes[1].scatter(X_pca[y_test == 0, 0], X_pca[y_test == 0, 1],
                   alpha=0.3, s=20, label='Normal', c='blue')
    axes[1].scatter(X_pca[y_test == 1, 0], X_pca[y_test == 1, 1],
                   alpha=0.3, s=20, label='Fault', c='orange')
    
    # Highlight errors
    if fp_mask.sum() > 0:
        axes[1].scatter(X_pca[fp_mask, 0], X_pca[fp_mask, 1],
                       alpha=0.9, s=100, label='False Positive', 
                       c='red', marker='x', linewidths=2)
    if fn_mask.sum() > 0:
        axes[1].scatter(X_pca[fn_mask, 0], X_pca[fn_mask, 1],
                       alpha=0.9, s=100, label='False Negative',
                       c='purple', marker='x', linewidths=2)
    
    axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
    axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
    axes[1].set_title('PCA: True Labels with Error Types', fontweight='bold', fontsize=12)
    axes[1].legend(loc='best')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nPCA Explained Variance:")
    print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
    print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
    print(f"  Total: {pca.explained_variance_ratio_.sum():.2%}")

## 5. Nearest Neighbor Investigation

In [ ]:
if df_pred is not None:
    # Select a few high-confidence errors for investigation
    high_conf_error_samples = df_pred[~df_pred['correct'] & (df_pred['confidence'] > 0.7)].head(3)
    
    if len(high_conf_error_samples) > 0:
        # Preprocess data
        config = PipelineConfig(standardize=True, use_pca=True)
        preprocessor = TimeSeriesPreprocessor(config)
        
        X_train_proc = preprocessor.fit_transform(X_train)
        X_test_proc = preprocessor.transform(X_test)
        
        print("\nInvestigating High-Confidence Errors:")
        print("="*80)
        
        for idx, row in high_conf_error_samples.iterrows():
            sample_id = row['sample_id']
            
            print(f"\nSample {sample_id}:")
            print(f"  True Label: {row['true_label']} ({'Normal' if row['true_label'] == 0 else 'Fault'})")
            print(f"  Predicted: {row['predicted_label']} ({'Normal' if row['predicted_label'] == 0 else 'Fault'})")
            print(f"  Confidence: {row['confidence']:.2%}")
            print(f"  Error Type: {row['error_type']}")
            
            # Find nearest neighbors
            query = X_test_proc[sample_id:sample_id+1]
            distances = np.sqrt(((X_train_proc - query) ** 2).sum(axis=(1, 2)))
            nn_indices = np.argsort(distances)[:10]
            
            print(f"\n  Nearest 10 Neighbors:")
            for i, nn_idx in enumerate(nn_indices, 1):
                nn_label = 'Normal' if y_train[nn_idx] == 0 else 'Fault'
                print(f"    {i:2d}. Distance: {distances[nn_idx]:.4f} | Label: {nn_label}")
            
            # Count labels in k nearest neighbors
            k_values = [5, 10, 15, 20]
            print(f"\n  Label distribution in k nearest neighbors:")
            for k in k_values:
                nn_labels = y_train[nn_indices[:k]]
                normal_count = (nn_labels == 0).sum()
                fault_count = (nn_labels == 1).sum()
                print(f"    k={k:2d}: Normal={normal_count:2d} ({normal_count/k:.1%}), "
                      f"Fault={fault_count:2d} ({fault_count/k:.1%})")

## 6. Recommendations for Improvement

In [ ]:
if df_pred is not None:
    # Summarize findings
    n_fp = (df_pred['error_type'] == 'False Positive').sum()
    n_fn = (df_pred['error_type'] == 'False Negative').sum()
    
    avg_conf_fp = df_pred[df_pred['error_type'] == 'False Positive']['confidence'].mean()
    avg_conf_fn = df_pred[df_pred['error_type'] == 'False Negative']['confidence'].mean()
    
    print("\nError Analysis Summary:")
    print("="*80)
    print(f"False Positives: {n_fp} (avg confidence: {avg_conf_fp:.2%})")
    print(f"False Negatives: {n_fn} (avg confidence: {avg_conf_fn:.2%})")
    print(f"\nTotal Errors: {n_fp + n_fn}")
    print(f"Error Rate: {(n_fp + n_fn) / len(df_pred):.2%}")
    
    print("\n" + "="*80)
    print("RECOMMENDATIONS FOR IMPROVEMENT")
    print("="*80)
    
    recommendations = []
    
    # Based on error patterns
    if n_fp > n_fn * 1.5:
        recommendations.append(
            "1. HIGH FALSE POSITIVE RATE:\n"
            "   - Consider adjusting decision threshold to reduce false alarms\n"
            "   - May need more diverse normal operation training data\n"
            "   - Consider class-weighted KNN or cost-sensitive learning"
        )
    elif n_fn > n_fp * 1.5:
        recommendations.append(
            "1. HIGH FALSE NEGATIVE RATE (Missed Faults):\n"
            "   - Critical for safety - consider lowering decision threshold\n"
            "   - May need more fault examples in training data\n"
            "   - Consider ensemble methods to improve recall"
        )
    else:
        recommendations.append(
            "1. BALANCED ERROR TYPES:\n"
            "   - Errors are relatively balanced between FP and FN\n"
            "   - Focus on improving overall discriminability"
        )
    
    # Based on confidence
    high_conf_errors = df_pred[~df_pred['correct'] & (df_pred['confidence'] > 0.7)]
    if len(high_conf_errors) > 0.05 * len(df_pred):
        recommendations.append(
            "\n2. HIGH CONFIDENCE ERRORS DETECTED:\n"
            f"   - {len(high_conf_errors)} errors with >70% confidence\n"
            "   - These are 'hard' cases near decision boundary\n"
            "   - Consider:\n"
            "     * Using DTW distance for better temporal alignment\n"
            "     * Feature engineering (gradients, trends, frequency domain)\n"
            "     * Larger k values for more robust neighborhoods"
        )
    
    recommendations.append(
        "\n3. GENERAL IMPROVEMENTS:\n"
        "   - Try alternative distance metrics (DTW, LCSS)\n"
        "   - Experiment with feature extraction (statistics, shapellets)\n"
        "   - Consider ensemble methods (Random Forest, XGBoost)\n"
        "   - Explore deep learning (LSTM, CNN) for automatic feature learning\n"
        "   - Add domain-specific features based on CSTH physics"
    )
    
    recommendations.append(
        "\n4. DATA QUALITY:\n"
        "   - Review misclassified samples for data quality issues\n"
        "   - Check for label noise or ambiguous cases\n"
        "   - Consider active learning to label borderline cases"
    )
    
    for rec in recommendations:
        print(rec)

## Summary

This error analysis revealed:

1. **Error Patterns**: Distribution and characteristics of misclassifications
2. **Confidence Analysis**: Many errors occur at moderate confidence levels
3. **Temporal Differences**: Error cases show different temporal patterns
4. **Feature Space**: Errors often occur at class boundaries in feature space
5. **Neighbor Analysis**: Some errors have mixed-class neighborhoods

**Key Insights**:
- High-confidence errors suggest some truly ambiguous cases
- Low-confidence errors indicate model uncertainty (good)
- Feature engineering could help separate difficult cases
- Alternative distance metrics may capture patterns better

**Next Steps**:
- Implement recommended improvements
- Test with different distance metrics
- Consider ensemble approaches
- Collect more training data for borderline cases